# [03] CAS(ME)² — Ekstraksi Fitur POC-ABS

Fitur POC-ABS: baseline = frame pertama klip, 3 ROI (`mulut`, `mata_kiri`, `mata_kanan`),
`block_size=7`, tiap blok menghasilkan `x`, `y`, `t`, `m` → 392 kolom.

**Face alignment dipakai di sini juga.** ROI diambil lewat `cl.FaceAligner` dengan frame
pertama klip sebagai referensi — frame yang sama dengan baseline POC. Ini wajib agar fitur
dihitung dengan cara yang sama seperti sinyal spotting di `[01]`; kalau tidak, jitter kotak
ROI akan masuk ke fitur dan mencemari input TabR.

Menghasilkan **dua file terpisah**:

| file | isi | peran |
|---|---|---|
| `features/poc_abs_gt.xlsx` | 57 klip ground truth dari `selectedpic` | data berlabel — dipakai melatih TabR |
| `features/poc_abs_spotted.xlsx` | semua klip hasil spotting sendiri | kondisi nyata — dipakai menguji TabR |

Kolom `status` di file kedua menandai `TP` (cocok GT, jadi emosinya diketahui) atau
`FP` (tidak cocok GT mana pun, diberi label `non_me`). Notebook `[04]` bisa menyaring
berdasarkan kolom ini tanpa perlu ekstraksi ulang.

In [1]:
import gc
import os
import sys
import time
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'casme2':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / 'casme2') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'casme2'))

from features_extraction.poc import POC
from features_extraction.quadran import Quadran
from features_extraction.vektor import Vektor

import casme2_lib as cl

# Harus sama dengan [01], kalau tidak path output dan cara hitung fitur akan berbeda.
config = cl.Casme2Config(project_root=PROJECT_ROOT, face_alignment=True)
cl.ensure_dir(config.feature_root)

MAX_WORKERS = min(4, os.cpu_count() or 1)
GT_FEATURE_PATH = config.feature_root / 'poc_abs_gt.xlsx'
SPOTTED_FEATURE_PATH = config.feature_root / 'poc_abs_spotted.xlsx'

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'FEATURE_ROOT = {config.feature_root}')
print(f'MAX_WORKERS  = {MAX_WORKERS}')
print(f'FACE_ALIGN   = {config.face_alignment}')
print(f'REGIONS      = {list(config.regions)}')

PROJECT_ROOT = /home/ryuko/skripsi/Skripsi/Convat-1st
FEATURE_ROOT = /home/ryuko/skripsi/Skripsi/Convat-1st/output/casme2/features_aligned
MAX_WORKERS  = 4
FACE_ALIGN   = True
REGIONS      = ['mulut', 'mata_kiri', 'mata_kanan']


## Daftar klip

Dua sumber, skema kolom identik supaya `[04]` bisa memperlakukan keduanya dengan kode yang sama.

In [2]:
ground_truth = cl.load_ground_truth(config, micro_only=True)

# --- sumber A: klip ground truth dari selectedpic ---
gt_clips = []
for row in ground_truth.itertuples(index=False):
    clip_dir = Path(row.selected_dir)
    if not clip_dir.exists():
        print(f'lewati {row.subject}/{row.clip_name}: folder selectedpic tidak ada')
        continue
    gt_clips.append({
        'source': 'gt',
        'status': 'GT',
        'participant': row.subject,
        'subject': row.subject,
        'video_code': row.video_code,
        'clip': row.video,
        'event_clip': row.clip_name,
        'event_no': 0,
        'label': row.emotion,
        'self_emotion': row.self_emotion,
        'onset_img': row.onset,
        'apex_img': row.apex,
        'offset_img': row.offset,
        'clip_dir': str(clip_dir),
    })

print(f'Klip ground truth (selectedpic): {len(gt_clips)}')
print(pd.DataFrame(gt_clips)['label'].value_counts().to_string())

Klip ground truth (selectedpic): 57
label
negative    21
others      21
positive    15


In [3]:
# --- sumber B: klip hasil spotting sendiri ---
matches_path = config.eval_root / 'iou_matches.xlsx'
if not config.spotting_metadata_path.exists():
    raise FileNotFoundError(f'Jalankan [01] dulu: {config.spotting_metadata_path}')
if not matches_path.exists():
    raise FileNotFoundError(f'Jalankan [02] dulu: {matches_path}')

metadata = pd.read_excel(config.spotting_metadata_path)
matches = pd.read_excel(matches_path)

status_lookup = matches.set_index(['video_key', 'event_clip'])[
    ['status', 'matched_clip_name', 'matched_emotion', 'matched_iou']
].to_dict('index')

spotted_clips = []
for row in metadata.itertuples(index=False):
    info = status_lookup.get((row.video_key, row.event_clip), {})
    status = info.get('status', 'FP')
    emotion = info.get('matched_emotion')
    spotted_clips.append({
        'source': 'spotted',
        'status': status,
        'participant': row.subject,
        'subject': row.subject,
        'video_code': row.video_code,
        'clip': row.video,
        'event_clip': row.event_clip,
        'event_no': int(row.event_no),
        # TP mewarisi emosi GT pasangannya; FP tidak punya emosi -> non_me
        'label': emotion if (status == 'TP' and isinstance(emotion, str)) else 'non_me',
        'self_emotion': None,
        'onset_img': int(row.onset_img),
        'apex_img': int(row.apex_img),
        'offset_img': int(row.offset_img),
        'matched_clip_name': info.get('matched_clip_name'),
        'matched_iou': info.get('matched_iou'),
        'clip_dir': str(config.output_root / row.clip_dir),
    })

spotted_df = pd.DataFrame(spotted_clips)
print(f'Klip hasil spotting: {len(spotted_clips)}')
print(spotted_df['status'].value_counts().to_string())
print()
print(spotted_df['label'].value_counts().to_string())

Klip hasil spotting: 1434
status
FP    1418
TP      16

label
non_me      1418
others         6
negative       5
positive       5


## Ekstraksi fitur

Satu baris per frame. Frame pertama tiap klip dipakai sebagai baseline sehingga tidak
menghasilkan baris sendiri — klip dengan kurang dari 2 frame otomatis dilewati dan dicatat.

In [4]:
WORKER_ALIGNER = None


def init_worker(config_dict: dict) -> None:
    # dlib tidak bisa di-pickle, jadi aligner dibangun di dalam worker.
    global WORKER_ALIGNER
    WORKER_ALIGNER = cl.FaceAligner(cl.Casme2Config(**config_dict))


META_FIELDS = [
    'source', 'status', 'participant', 'subject', 'video_code', 'clip', 'event_clip',
    'event_no', 'label', 'self_emotion', 'onset_img', 'apex_img', 'offset_img',
    'matched_clip_name', 'matched_iou',
]


def extract_clip(clip: dict) -> dict:
    # POC-ABS satu klip. Baseline = frame pertama, yang sekaligus jadi referensi alignment.
    clip_dir = Path(clip['clip_dir'])
    try:
        frame_paths = cl.load_frame_paths(clip_dir)
        if len(frame_paths) < 2:
            return {'ok': False, 'rows': [], 'error': f'butuh >= 2 frame, ada {len(frame_paths)}',
                    'event_clip': clip['event_clip']}

        # Referensi alignment dipasang ulang untuk tiap klip.
        WORKER_ALIGNER.reset()
        block_size = WORKER_ALIGNER.config.block_size

        baseline = cv2.imread(str(frame_paths[0]))
        if baseline is None:
            return {'ok': False, 'rows': [], 'error': 'gagal baca baseline',
                    'event_clip': clip['event_clip']}
        baseline_rois = WORKER_ALIGNER.rois(baseline)
        if baseline_rois is None:
            return {'ok': False, 'rows': [], 'error': 'wajah tidak terdeteksi di baseline',
                    'event_clip': clip['event_clip']}

        meta = {k: clip.get(k) for k in META_FIELDS}
        meta['event_id'] = f"{clip['source']}||{clip['participant']}||{clip['clip']}||{clip['event_clip']}"

        rows = []
        for frame_no, frame_path in enumerate(frame_paths[1:], start=2):
            image = cv2.imread(str(frame_path))
            if image is None:
                continue
            rois = WORKER_ALIGNER.rois(image)
            if rois is None:
                continue

            row = dict(meta)
            row['frame'] = frame_no
            for region_name in WORKER_ALIGNER.config.regions:
                poc = POC(baseline_rois[region_name], rois[region_name], block_size)
                vec = Vektor(poc.getPOC(), block_size)
                quad = Quadran(vec.getVektor()).getQuadran()
                for block_id, block in enumerate(quad, start=1):
                    row[f'{region_name}_x{block_id}'] = block[1]
                    row[f'{region_name}_y{block_id}'] = block[2]
                    row[f'{region_name}_t{block_id}'] = block[3]
                    row[f'{region_name}_m{block_id}'] = block[4]
            rows.append(row)
        return {'ok': True, 'rows': rows, 'error': None, 'event_clip': clip['event_clip']}
    except Exception as exc:
        return {'ok': False, 'rows': [], 'error': str(exc), 'event_clip': clip['event_clip']}


worker_config = {
    'project_root': str(PROJECT_ROOT),
    'face_alignment': config.face_alignment,
    'regions': config.regions,
    'target_size': {k: tuple(v) for k, v in config.target_size.items()},
    'padding_x': config.padding_x,
    'padding_y': config.padding_y,
    'block_size': config.block_size,
}


def run_extraction(clips: list[dict], name: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    start = time.time()
    rows, failures = [], []
    with ProcessPoolExecutor(max_workers=MAX_WORKERS, initializer=init_worker,
                             initargs=(worker_config,)) as executor:
        futures = {executor.submit(extract_clip, clip): clip for clip in clips}
        for done, future in enumerate(as_completed(futures), start=1):
            result = future.result()
            if result['ok']:
                rows.extend(result['rows'])
            else:
                failures.append({'event_clip': result['event_clip'], 'error': result['error']})
            if done % 50 == 0 or done == len(clips):
                print(f'  [{name}] {done}/{len(clips)} klip, {len(rows)} baris', flush=True)
    print(f'  [{name}] selesai dalam {time.time() - start:.0f} detik, gagal {len(failures)} klip')
    return pd.DataFrame(rows), pd.DataFrame(failures)

### File 1 — klip ground truth (`selectedpic`)

In [5]:
gt_features, gt_failures = run_extraction(gt_clips, 'GT')
gt_features.to_excel(GT_FEATURE_PATH, index=False)

feature_cols = [c for c in gt_features.columns
                if c not in META_FIELDS + ['event_id', 'frame']]

print(f'\nBaris          : {len(gt_features)}')
print(f'Event unik     : {gt_features["event_id"].nunique()}')
print(f'Kolom fitur    : {len(feature_cols)}')
print(f'Disimpan       : {GT_FEATURE_PATH}')
print()
print(gt_features.groupby('label')['event_id'].nunique().to_string())
if len(gt_failures):
    display(gt_failures)
gt_features[['participant', 'clip', 'event_clip', 'label', 'frame']].head()

  [GT] 50/57 klip, 628 baris
  [GT] 57/57 klip, 717 baris
  [GT] selesai dalam 7 detik, gagal 0 klip

Baris          : 717
Event unik     : 57
Kolom fitur    : 392
Disimpan       : /home/ryuko/skripsi/Skripsi/Convat-1st/output/casme2/features_aligned/poc_abs_gt.xlsx

label
negative    21
others      21
positive    15


,participant,clip,event_clip,label,frame
0,s15,15_0102eatingworms,disgust2_3,negative,2
1,s15,15_0102eatingworms,disgust2_3,negative,3
2,s15,15_0102eatingworms,disgust2_3,negative,4
3,s15,15_0102eatingworms,disgust2_3,negative,5
4,s15,15_0102eatingworms,disgust2_3,negative,6


### File 2 — klip hasil spotting sendiri

In [6]:
spotted_features, spotted_failures = run_extraction(spotted_clips, 'SPOTTED')
spotted_features.to_excel(SPOTTED_FEATURE_PATH, index=False)

print(f'\nBaris          : {len(spotted_features)}')
print(f'Event unik     : {spotted_features["event_id"].nunique()}')
print(f'Disimpan       : {SPOTTED_FEATURE_PATH}')
print()
print('--- per status ---')
print(spotted_features.groupby('status')['event_id'].nunique().to_string())
print()
print('--- per label ---')
print(spotted_features.groupby('label')['event_id'].nunique().to_string())
if len(spotted_failures):
    display(spotted_failures.head(20))
spotted_features[['participant', 'clip', 'event_clip', 'status', 'label', 'frame']].head()

  [SPOTTED] 50/1434 klip, 1027 baris
  [SPOTTED] 100/1434 klip, 1941 baris
  [SPOTTED] 150/1434 klip, 2781 baris
  [SPOTTED] 200/1434 klip, 3753 baris
  [SPOTTED] 250/1434 klip, 4715 baris
  [SPOTTED] 300/1434 klip, 5676 baris
  [SPOTTED] 350/1434 klip, 6662 baris
  [SPOTTED] 400/1434 klip, 7563 baris
  [SPOTTED] 450/1434 klip, 8453 baris
  [SPOTTED] 500/1434 klip, 9389 baris
  [SPOTTED] 550/1434 klip, 10410 baris
  [SPOTTED] 600/1434 klip, 11370 baris
  [SPOTTED] 650/1434 klip, 12280 baris
  [SPOTTED] 700/1434 klip, 13281 baris
  [SPOTTED] 750/1434 klip, 14229 baris
  [SPOTTED] 800/1434 klip, 15172 baris
  [SPOTTED] 850/1434 klip, 16072 baris
  [SPOTTED] 900/1434 klip, 16961 baris
  [SPOTTED] 950/1434 klip, 17902 baris
  [SPOTTED] 1000/1434 klip, 18882 baris
  [SPOTTED] 1050/1434 klip, 19671 baris
  [SPOTTED] 1100/1434 klip, 20729 baris
  [SPOTTED] 1150/1434 klip, 21724 baris
  [SPOTTED] 1200/1434 klip, 22755 baris
  [SPOTTED] 1250/1434 klip, 23557 baris
  [SPOTTED] 1300/1434 klip, 24

,participant,clip,event_clip,status,label,frame
0,s15,15_0102eatingworms,event_00388-00404,FP,non_me,2
1,s15,15_0102eatingworms,event_00388-00404,FP,non_me,3
2,s15,15_0102eatingworms,event_00388-00404,FP,non_me,4
3,s15,15_0102eatingworms,event_00388-00404,FP,non_me,5
4,s15,15_0102eatingworms,event_00388-00404,FP,non_me,6


## Ringkasan

In [7]:
overview = pd.DataFrame([
    {'file': GT_FEATURE_PATH.name, 'baris': len(gt_features),
     'event': gt_features['event_id'].nunique(),
     'kolom_fitur': len(feature_cols), 'peran': 'training TabR'},
    {'file': SPOTTED_FEATURE_PATH.name, 'baris': len(spotted_features),
     'event': spotted_features['event_id'].nunique(),
     'kolom_fitur': len([c for c in spotted_features.columns
                         if c not in META_FIELDS + ['event_id', 'frame']]),
     'peran': 'pengujian end-to-end'},
])
print(overview.to_string(index=False))
print(f'\nKolom meta: {META_FIELDS + ["event_id", "frame"]}')
print('\nLanjut ke [04] untuk training TabR LOSO.')
overview

                file  baris  event  kolom_fitur                peran
     poc_abs_gt.xlsx    717     57          392        training TabR
poc_abs_spotted.xlsx  26747   1434          392 pengujian end-to-end

Kolom meta: ['source', 'status', 'participant', 'subject', 'video_code', 'clip', 'event_clip', 'event_no', 'label', 'self_emotion', 'onset_img', 'apex_img', 'offset_img', 'matched_clip_name', 'matched_iou', 'event_id', 'frame']

Lanjut ke [04] untuk training TabR LOSO.


,file,baris,event,kolom_fitur,peran
0,poc_abs_gt.xlsx,717,57,392,training TabR
1,poc_abs_spotted.xlsx,26747,1434,392,pengujian end-to-end
